# LangChain RAG Tool — Rewritten for LangChain v1 (`langchain==1.3.11`)

This notebook keeps the **exact same task** as the original `9_LangchainRAGTool.ipynb`: build a FAISS-backed retriever over `sample.txt`, wrap it as a tool the agent can call, give the agent conversational memory, and ask it the same 3 questions — *"What is LangChain?"*, *"Who created it?"*, *"Explain LangChain's use in AI workflows."*

Only **how** it's built has changed — the legacy `initialize_agent` + `AgentType.CONVERSATIONAL_REACT_DESCRIPTION` + `RetrievalQA` + `ConversationBufferMemory` stack is replaced with [`create_agent`](https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent):

| Legacy piece | v1 replacement |
|---|---|
| `RetrievalQA.from_chain_type(llm, retriever)` wrapped in a `Tool` | A plain `@tool` function that calls `retriever.invoke(query)` and has the LLM answer from that context |
| `ConversationBufferMemory(memory_key="chat_history", ...)` | `checkpointer=InMemorySaver()` + `thread_id` |
| `initialize_agent(..., agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, handle_parsing_errors=True)` | `create_agent(model, tools, checkpointer=...)` |
| `handle_parsing_errors=True` (needed because the old text-based ReAct agent could fail to parse the LLM's free-form output — see the "Could not parse LLM output" error in the original notebook's own Q3 run) | Not needed — native tool calling has no free-text format to fail on |
| `langchain.chat_models.ChatOpenAI` | `langchain_openai.ChatOpenAI` |
| `langchain.vectorstores.FAISS` / `langchain.embeddings.OpenAIEmbeddings` | `langchain_community.vectorstores.FAISS` / `langchain_openai.OpenAIEmbeddings` |
| `langchain.text_splitter.CharacterTextSplitter` | `langchain_text_splitters.CharacterTextSplitter` |

Reference: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>


In [1]:
%pip install -U langchain-google-genai langchain-community langchain-text-splitters faiss-cpu langgraph python-dotenv

  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
Using cached langchain_text_splitters-1.1.3-py3-none-any.whl (38 kB)
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.6 MB 3.3 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/16.6 MB 4.4 MB/s eta 0:00:04
   ----- ---------------------------------- 2.4/16.6 MB 4.3 MB/s eta 0:00:04
   -------- ------------------------------- 3.7/16.6 MB 4.8 MB/s eta 0:00:03
   ----------- ---------------------------- 5.0/16.6 MB 4.9 MB/s eta 0:00:03
   -------------- ------------------------- 6.0/16.6 MB 5.1 MB/s eta 0:00:03
   ----------------- ---------------------- 7.3/16.6 MB 5.3 MB/s eta 0:00:02
   -------------------- ------------------- 8.7/16.6 MB 5.4 MB/s eta 0:00:02
   ---------------------- ----------------- 9.2/16.6 MB 5.4 MB/s eta 0:00:02
   --------------

In [1]:

# 1) Imports
from langchain.agents import create_agent
from langchain.tools import tool

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

import os
from dotenv import load_dotenv


# 2) Load Gemini API key
load_dotenv(".env")

gemini_api_key = os.getenv("GEMINI_API_KEY")

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is missing in .env file")


# 3) Setup Gemini LLM (AI Brain)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=gemini_api_key
)


# 4) Read sample.txt
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()


# 5) Split text into smaller chunks
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_text(text_data)


# 6) Create Gemini Embeddings
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=gemini_api_key
)


# 7) Create FAISS Vector Database
vectorstore = FAISS.from_texts(
    texts,
    embedding
)

retriever = vectorstore.as_retriever()


# 8) Create Retriever Tool
@tool
def LangChainRetriever(query: str) -> str:
    """Use this tool to answer questions about LangChain framework, features, or its creator."""

    docs = retriever.invoke(query)

    if not docs:
        return "I couldn't find relevant information in sample.txt."

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    answer = llm.invoke(
        f"""
        Answer the question using only the context below.

        Context:
        {context}

        Question:
        {query}
        """
    )

    return answer.content


# 9) Setup Short-Term Memory
checkpointer = InMemorySaver()

thread_config = {
    "configurable": {
        "thread_id": "rag-demo-1"
    }
}


# 10) Create Gemini Agent with Tool and Memory
agent = create_agent(
    model=llm,
    tools=[LangChainRetriever],
    system_prompt=(
        "You are a helpful assistant. "
        "For questions about LangChain, use the LangChainRetriever tool. "
        "Use conversation history to understand follow-up questions."
    ),
    checkpointer=checkpointer
)


# 11) Ask First Question
print("1️⃣ First Question")

res1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is LangChain?"}
        ]
    },
    thread_config
)

print("Answer:", res1["messages"][-1].content)


# 12) Ask Follow-up Question
print("\n2️⃣ Follow-up")

res2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Who created it?"}
        ]
    },
    thread_config
)

print("Answer:", res2["messages"][-1].content)


# 13) Combined Reasoning
print("\n3️⃣ Combined Reasoning")

res3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Explain LangChain's use in AI workflows."
            }
        ]
    },
    thread_config
)

print("Answer:", res3["messages"][-1].content)


C:\Users\gnana\AppData\Local\Temp\ipykernel_24432\1021273142.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


1️⃣ First Question
Answer: LangChain is a framework designed for building applications that use Large Language Models (LLMs).

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase.

3️⃣ Combined Reasoning
Answer: LangChain is commonly used in AI workflows.
